# POI Incremental Load Pipeline

## Overview
This notebook processes **incremental POI CSV uploads** through the medallion architecture: **Bronze (append) → Silver (MERGE) → Gold (MERGE)**. It is designed to run after the initial full load has been completed via the [Data Ingestion](/#notebook/1915875470418234), [POI EDA & QA](/#notebook/3304086644673254), and [POI Data Transformation](/#notebook/3304086644673256) notebooks.

## Dual-Mode Execution
| Condition | Behaviour |
| --- | --- |
| `delta.csv` **exists** in S3 | Run full incremental pipeline (QA → Bronze → Silver → Gold), then load all gold tables |
| `delta.csv` **does not exist** | Skip transformations, load existing gold tables directly |

Either way, the notebook finishes with all gold dimension and fact tables loaded as Spark DataFrames.

## Pipeline Steps (when incremental data exists)
| Step | Layer | Operation | Description |
| --- | --- | --- | --- |
| 1 | **Bronze** | Append | New CSV rows are appended to `edbi_teamg02.bronze.poi` (raw audit trail) |
| 2 | **Silver** | MERGE (upsert) | Deduplicated, QA'd rows are merged into `edbi_teamg02.silver.poi` |
| 3 | **Gold** | MERGE (upsert) | Enriched rows are merged into `edbi_teamg02.gold.fact_poi_enriched` and `edbi_teamg02.gold.dim_poi` |
| 4 | **Gold** | UPDATE | `edbi_teamg02.gold.dim_employer.poi_per_employer` is recalculated |

## Key Design Decisions
* **Primary key**: `NRIC` — each POI is uniquely identified by their NRIC
* **MERGE logic**: If NRIC already exists and new `Year of Profile` >= existing → **UPDATE**; otherwise → **INSERT** new rows
* **Employer & Industry tables are unchanged** — only POI data is incremental
* **Idempotent**: Safe to run even when no new data is available

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/marcus_lim@cpib.gov.sg")

import pandas as pd
import numpy as np
from poi_utils import clean_nulls, qa_poi, enrich_poi, prepare_for_spark

# Configuration — fixed incremental filename
S3_PATH = "s3://edbi-03/s3_teamG02/delta.csv"

# Check if delta.csv exists in S3
try:
    dbutils.fs.ls(S3_PATH)
    HAS_INCREMENTAL = True
    print(f"✅ Incremental file found: {S3_PATH}")
    print("   → Running full incremental pipeline")
except Exception:
    HAS_INCREMENTAL = False
    print(f"⚠️  No incremental file at: {S3_PATH}")
    print("   → Skipping transformations, loading existing gold tables")

✅ Incremental file found: s3://edbi-03/s3_teamG02/delta.csv
   → Running full incremental pipeline


In [0]:
if HAS_INCREMENTAL:
    df_incr_spark = (spark.read.format("csv")
        .option("inferSchema", "true")
        .option("header", "true")
        .load(S3_PATH))

    df_incr = df_incr_spark.toPandas()

    print(f"Loaded {df_incr.shape[0]} rows, {df_incr.shape[1]} columns from {S3_PATH}")
    print(f"\nColumns: {list(df_incr.columns)}")
    print(f"\nPreview:")
    display(df_incr.head())
else:
    print("Skipped — no incremental file.")

Loaded 10000 rows, 12 columns from s3://edbi-03/s3_teamG02/delta.csv

Columns: ['NRIC', 'Name', 'Race', 'Gender', 'Email', 'Marital Status', 'Nationality', 'DOB', 'Phone Number', 'Occupation', 'UEN Identifier', 'Integer']

Preview:


NRIC,Name,Race,Gender,Email,Marital Status,Nationality,DOB,Phone Number,Occupation,UEN Identifier,Integer
S9582940Q,Gim Hock Lim,Chinese,M,dennis44@example.com,Divorced/Separated,Greek,15/3/1954,8.1426362E7,Production and Specialised Services Managers,01230000D,2000.0
S9953701W,Alvin Lin,Chinese,M,yfuentes@example.org,Married,Irish,none,9.72575E7,"Hospitality, Retail & Related Services Managers",NIL,2000.0
S4847253E,Nur Binte Khalid,Malay,F,herringrobert@example.org,Single,Solomon Islander,1/12/1995,9.4958808E7,Business & Administration Professionals,00445300W,2000.0
S6868599R,Liang Hong Chng,Chinese,M,brownchristine@example.net,Married,Egyptian,none,8.7700704E7,Agricultural & Fishery Workers,00819900X,2000.0
S7150118T,Venkatesh S/O Anand,Indian,M,josephjessica@example.net,Single,Azerbaijani,20/10/1989,9.0457212E7,Business & Administration Associate Professionals,04746000L,2000.0


In [0]:
if HAS_INCREMENTAL:
    from pyspark.sql.types import IntegerType

    # Append raw (pre-QA) data to bronze — preserves audit trail
    # Cast int64 → int32 to match existing bronze schema (Phone Number INT, Integer INT)
    df_bronze_append = df_incr_spark
    for col_name in ["Phone Number", "Integer"]:
        if col_name in df_bronze_append.columns:
            df_bronze_append = df_bronze_append.withColumn(
                col_name, df_bronze_append[f"`{col_name}`"].cast(IntegerType())
            )

    (df_bronze_append.write.format("delta")
        .mode("append")
        .option("delta.columnMapping.mode", "name")
        .saveAsTable("edbi_teamg02.bronze.poi"))

    print(f"Appended {df_bronze_append.count()} rows to edbi_teamg02.bronze.poi")
else:
    print("Skipped — no incremental file.")

Appended 10000 rows to edbi_teamg02.bronze.poi


In [0]:
if HAS_INCREMENTAL:
    # Apply QA on the pandas copy (renames Integer → Year of Profile, dedupes, etc.)
    df_incr = qa_poi(df_incr)

    # Create QA'd Spark DataFrame for downstream silver/gold MERGE
    df_incr_spark_clean = spark.createDataFrame(df_incr)
else:
    print("Skipped — no incremental file.")

  After removing null NRICs: 95 rows
  After dedup by NRIC: 95 rows
  After DOB validation: 80 rows
  QA complete: 10000 → 80 rows (9920 removed)


In [0]:
if HAS_INCREMENTAL:
    # Create temp view from the QA'd incremental data
    df_incr_spark_clean.createOrReplaceTempView("incremental_poi_silver")

    # Track counts before merge
    before_count = spark.sql("SELECT COUNT(*) AS cnt FROM edbi_teamg02.silver.poi").collect()[0]["cnt"]

    # Execute MERGE
    spark.sql("""
        MERGE INTO edbi_teamg02.silver.poi AS target
        USING incremental_poi_silver AS source
        ON target.NRIC = source.NRIC
        WHEN MATCHED AND source.`Year of Profile` >= target.`Year of Profile` THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)

    # Track counts after merge
    after_count = spark.sql("SELECT COUNT(*) AS cnt FROM edbi_teamg02.silver.poi").collect()[0]["cnt"]

    print(f"Silver MERGE complete:")
    print(f"  Before: {before_count} rows")
    print(f"  After:  {after_count} rows")
    print(f"  Net new rows: {after_count - before_count}")
    print(f"  Updated rows: {len(df_incr) - (after_count - before_count)}")
else:
    print("Skipped — no incremental file.")

Silver MERGE complete:
  Before: 7322 rows
  After:  7322 rows
  Net new rows: 0
  Updated rows: 80


In [0]:
if HAS_INCREMENTAL:
    # Get the list of NRICs from the incremental batch
    incremental_nrics = set(df_incr["NRIC"].unique())

    # Read silver POI — filter to only incremental NRICs
    df_poi = spark.read.table("edbi_teamg02.silver.poi").toPandas()
    df_poi = df_poi[df_poi["NRIC"].isin(incremental_nrics)].copy()

    # Read employer and industry in full (needed for joins)
    df_employer = spark.read.table("edbi_teamg02.silver.employer").toPandas()
    df_industry = spark.read.table("edbi_teamg02.silver.industry").toPandas()

    print(f"df_poi (incremental NRICs only): {df_poi.shape}")
    print(f"df_employer: {df_employer.shape}")
    print(f"df_industry: {df_industry.shape}")
else:
    print("Skipped — no incremental file.")

df_poi (incremental NRICs only): (80, 12)
df_employer: (2700, 17)
df_industry: (988, 11)


In [0]:
if HAS_INCREMENTAL:
    df_enriched = enrich_poi(df_poi, df_employer, df_industry)
else:
    print("Skipped — no incremental file.")

  Enriched: 80 rows, 53 cols
  Employer matched: 64 / 80
  Industry matched: 63 / 80


In [0]:
if HAS_INCREMENTAL:
    df_fact = prepare_for_spark(df_enriched)
    spark_fact = spark.createDataFrame(df_fact)
    spark_fact.createOrReplaceTempView("incremental_fact")

    fact_before = spark.sql("SELECT COUNT(*) AS cnt FROM edbi_teamg02.gold.fact_poi_enriched").collect()[0]["cnt"]

    spark.sql("""
        MERGE INTO edbi_teamg02.gold.fact_poi_enriched AS target
        USING incremental_fact AS source
        ON target.nric = source.nric
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    fact_after = spark.sql("SELECT COUNT(*) AS cnt FROM edbi_teamg02.gold.fact_poi_enriched").collect()[0]["cnt"]
    print(f"Gold fact_poi_enriched MERGE: {fact_before} → {fact_after} rows (+{fact_after - fact_before} new, {len(df_fact) - (fact_after - fact_before)} updated)")
else:
    print("Skipped — no incremental file.")

Gold fact_poi_enriched MERGE: 7322 → 7322 rows (+0 new, 80 updated)


In [0]:
if HAS_INCREMENTAL:
    poi_dim_cols = [
        "nric", "name", "race", "gender", "email", "marital_status",
        "nationality", "nationality_region", "dob", "phone_number",
        "occupation", "occupation_sector", "occupation_skill_level",
        "year_of_profile", "age", "age_group",
        "uen_identifier", "has_employer", "is_unemployed"
    ]
    df_dim_poi = prepare_for_spark(df_enriched[poi_dim_cols].drop_duplicates(subset=["nric"]))
    spark_dim_poi = spark.createDataFrame(df_dim_poi)
    spark_dim_poi.createOrReplaceTempView("incremental_dim_poi")

    dim_before = spark.sql("SELECT COUNT(*) AS cnt FROM edbi_teamg02.gold.dim_poi").collect()[0]["cnt"]

    spark.sql("""
        MERGE INTO edbi_teamg02.gold.dim_poi AS target
        USING incremental_dim_poi AS source
        ON target.nric = source.nric
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    dim_after = spark.sql("SELECT COUNT(*) AS cnt FROM edbi_teamg02.gold.dim_poi").collect()[0]["cnt"]
    print(f"Gold dim_poi MERGE: {dim_before} → {dim_after} rows (+{dim_after - dim_before} new)")
else:
    print("Skipped — no incremental file.")

Gold dim_poi MERGE: 7322 → 7322 rows (+0 new)


In [0]:
if HAS_INCREMENTAL:
    spark.sql("""
        WITH employer_counts AS (
            SELECT uen_identifier, COUNT(DISTINCT nric) AS poi_per_employer
            FROM edbi_teamg02.gold.fact_poi_enriched
            WHERE has_employer = true
            GROUP BY uen_identifier
        )
        MERGE INTO edbi_teamg02.gold.dim_employer AS target
        USING employer_counts AS source
        ON target.uen_identifier = source.uen_identifier
        WHEN MATCHED THEN
            UPDATE SET target.poi_per_employer = source.poi_per_employer
    """)
    print("Gold dim_employer poi_per_employer recalculated.")
else:
    print("Skipped — no incremental file.")

Gold dim_employer poi_per_employer recalculated.


In [0]:
# Always load gold tables regardless of incremental path
gold_fact = spark.read.table("edbi_teamg02.gold.fact_poi_enriched")
gold_dim_poi = spark.read.table("edbi_teamg02.gold.dim_poi")
gold_dim_employer = spark.read.table("edbi_teamg02.gold.dim_employer")
gold_dim_industry = spark.read.table("edbi_teamg02.gold.dim_industry")

print("=" * 60)
print("PIPELINE SUMMARY")
print("=" * 60)
print(f"Mode: {'INCREMENTAL' if HAS_INCREMENTAL else 'NO-OP (existing data)'}")
print(f"\n{'Table':<50} {'Rows':>8}")
print("-" * 60)
for name, df in [("edbi_teamg02.gold.fact_poi_enriched", gold_fact),
                  ("edbi_teamg02.gold.dim_poi", gold_dim_poi),
                  ("edbi_teamg02.gold.dim_employer", gold_dim_employer),
                  ("edbi_teamg02.gold.dim_industry", gold_dim_industry)]:
    print(f"{name:<50} {df.count():>8,}")
print("-" * 60)
print("\nAll gold tables loaded as: gold_fact, gold_dim_poi, gold_dim_employer, gold_dim_industry")

PIPELINE SUMMARY
Mode: INCREMENTAL

Table                                                  Rows
------------------------------------------------------------
edbi_teamg02.gold.fact_poi_enriched                   7,322
edbi_teamg02.gold.dim_poi                             7,322
edbi_teamg02.gold.dim_employer                        2,286
edbi_teamg02.gold.dim_industry                          988
------------------------------------------------------------

All gold tables loaded as: gold_fact, gold_dim_poi, gold_dim_employer, gold_dim_industry
